In [ ]:
import scanpy as sc
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../../../")
sys.path.append("../../utils/")
import mapqc
import analysis

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
dist_2 = "sinkhorn"  # sinkhorn or centroid

In [ ]:
path_alt_dist_results = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/Sheppard_2020/cellxgene_counts_batch_dataset_kl1.0/mapqc/control_{dist_2}.pkl"
path_pairwise_dist_results = "/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/Sheppard_2020/cellxgene_counts_batch_dataset_kl1.0/mapqc/control.pkl"

In [ ]:
res_pw = mapqc.load(path_pairwise_dist_results)
res_alt = mapqc.load(path_alt_dist_results)

In [ ]:
res_pw.center_cells

In [ ]:
res_alt.center_cells

In [ ]:
x = np.array(res_pw.adata.obs.mapqc_score.values)
y = np.array(res_alt.adata.obs.mapqc_score.values)
mask = pd.isnull(x) | pd.isnull(y)
x = x[~mask]
y = y[~mask]
margin = 0.5
diffs = x - y
x_higher_than_y = diffs > margin
x_lower_than_y = diffs < -margin
colors = [
    "red" if x_higher else "blue" if x_lower else "black"
    for x_higher, x_lower in zip(x_higher_than_y, x_lower_than_y)
]
fig, ax = plt.subplots()
ax.scatter(x, y, s=0.1, c=colors)
ax.set_xlabel("MapQC score, pairwise distance-based")
ax.set_ylabel(f"MapQC score, {dist_2} distance-based")
ax.set_xlim((-2, 10))
ax.set_ylim((-2, 10))
plt.show()

In [ ]:
print(
    f"Pairwise distance-based mapqc_scores are larger than {dist_2}-based in {sum(x_higher_than_y)} cases."
)

In [ ]:
print(
    f"Pairwise distance-based mapqc_scores are smaller than {dist_2}-based in {sum(x_lower_than_y)} cases."
)

Check difference per celltype:

In [ ]:
(res_pw.adata.obs.index == res_alt.adata.obs.index).all()

In [ ]:
celltype_score_df_pw = (
    res_pw.adata.obs.loc[:, ["ann_level_4_pred", "mapqc_score"]]
    .copy()
    .dropna(how="all")
)
celltype_score_df_pw["Distance type"] = "Pairwise euclidean"
celltype_score_df_alt = (
    res_alt.adata.obs.loc[:, ["ann_level_4_pred", "mapqc_score"]]
    .copy()
    .dropna(how="all")
)
celltype_score_df_alt["Distance type"] = "Sinkhorn"
celltype_score_df = pd.concat(
    (celltype_score_df_pw, celltype_score_df_alt), axis=0, ignore_index=True
)

In [ ]:
import importlib

In [ ]:
importlib.reload(analysis)

In [ ]:
celltype_score_df["ann_level_4_pred"] = analysis.clean_up_df_column(
    celltype_score_df, "ann_level_4_pred", min_n_cells_per_cat=200
)

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(
    celltype_score_df,
    x="ann_level_4_pred",
    y="mapqc_score",
    hue="Distance type",
    ax=ax,
    fliersize=0.2,
)
ax.set_title("MapQC scores pairwise vs. sinkhorn distance-based")
ax.tick_params(axis="x", rotation=90)
ax.set_ylabel("MapQC score")
plt.axhline(y=2, linestyle="--")
plt.show()

mapqc score:

### Results pairwise versus sinkhorn:

In [ ]:
mapqc.pl.umap_mapqc_scores(res_pw)

In [ ]:
mapqc.pl.umap_mapqc_scores(res_alt)

binarized mapqc score:

In [ ]:
mapqc.pl.umap_mapqc_scores(res_pw, color_by="mapqc_score>2")

In [ ]:
mapqc.pl.umap_mapqc_scores(res_alt, color_by="mapqc_score>2")

#### Look into specific neighborhoods:

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(res_pw, label_x_ticks_by="ann_level_4_pred")

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(res_alt, label_x_ticks_by="ann_level_4_pred")

In [ ]:
nhood_diffs = pd.DataFrame(
    res_pw.nhood_stats.median_mapqc_score
    - res_alt.nhood_stats.loc[res_pw.nhood_stats.index, "median_mapqc_score"]
)
nhood_diffs.rename(columns={"median_mapqc_score": "delta_mapqc_vs_alt"}, inplace=True)

In [ ]:
nhood_diffs["ann_level_4_pred"] = res_pw.adata.obs.loc[
    nhood_diffs.index, "ann_level_4_pred"
]
nhood_diffs["nhood_number"] = res_pw.nhood_stats.nhood_number

In [ ]:
nhood_diffs.groupby("ann_level_4_pred").agg({"delta_mapqc_vs_alt": "mean"})

In [ ]:
nhood_oi = res_pw.nhood_stats.index[res_pw.nhood_stats.nhood_number == 307][0]

In [ ]:
mapqc.pl.umap_center_cell(res_pw, neighborhood_center_cell=nhood_oi)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(
    res_pw, neighborhood=nhood_oi, use_zscored_dists=True
)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(
    res_alt, neighborhood=nhood_oi, use_zscored_dists=True
)

In [ ]:
res_pw

In [ ]:
mapqc.pl.umap_neighborhood(
    res_pw,
    color_by=["sample", "study", "r_or_q", "mapqc_score", "mapqc_score>2"],
    neighborhood=nhood_oi,
    size=20,
    recalculate_umap=True,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
)

In [ ]:
mapqc.pl.umap_neighborhood(
    res_alt,
    color_by=["sample", "study", "r_or_q", "mapqc_score", "mapqc_score>2"],
    neighborhood=nhood_oi,
    size=20,
    recalculate_umap=True,
    umap_kwargs={"cmap": "coolwarm", "vmin": -4, "vmax": 4},
)

plot actual latent dimensions, to avoid umap contortion:

In [ ]:
nhood_idx

In [ ]:
nhood_idx = np.where(res_pw.center_cells == nhood_oi)[0][0]
nhood_cells = res_pw.nhood_cells[nhood_idx, :]
# remove nans and set to integers instead of floats:
nhood_cells = nhood_cells[~np.isnan(nhood_cells)].astype(int)
nhood_adata = res_pw.adata[nhood_cells, :].copy()

In [ ]:
nhood_adata

In [ ]:
importlib.reload(mapqc.pl)

In [ ]:
res_pw.adata.obs["q_sample"] = "Reference"
res_pw.adata.obs.loc[res_pw.adata.obs.r_or_q == "q", "q_sample"] = res_pw.adata.obs.loc[
    res_pw.adata.obs.r_or_q == "q", "sample"
]

Calculate per sample variance:

In [ ]:
sample_var_per_nhood_pw = pd.DataFrame(
    index=res_pw.adata.obs["sample"].unique(), columns=res_pw.center_cells
)
for nhood in res_pw.center_cells:
    nhood_idx = np.where(res_pw.center_cells == nhood)[0][0]
    nhood_cells = res_pw.nhood_cells[nhood_idx, :]
    # remove nans and set to integers instead of floats:
    nhood_cells = nhood_cells[~np.isnan(nhood_cells)].astype(int)
    nhood_adata = res_pw.adata[nhood_cells, :].copy()
    for s in nhood_adata.obs["sample"].unique():
        s_var = np.var(nhood_adata[nhood_adata.obs["sample"] == s, :].X, axis=0).sum()
        sample_var_per_nhood_pw.loc[s, nhood] = s_var

check relation between increased sample variance in q compared to reference, versus large difference in mapqc score in sinkhorn versus pw:

In [ ]:
sample_to_rq = res_pw.adata.obs.groupby("sample").agg({"r_or_q": "first"})

In [ ]:
sample_var_per_nhood_pw["r_or_q"] = sample_to_rq.loc[
    sample_var_per_nhood_pw.index, "r_or_q"
]

In [ ]:
sample_var_ref_vs_q_pw = sample_var_per_nhood_pw.groupby("r_or_q").agg(
    {nhood: "mean" for nhood in res_pw.center_cells}
)

In [ ]:
sample_var_ratio_ref_vs_q_pw = (
    sample_var_ref_vs_q_pw.loc["q", :] / sample_var_ref_vs_q_pw.loc["r", :]
)

In [ ]:
sample_var_ratio_ref_vs_q_pw.sort_values()

In [ ]:
nhood_diffs["q_var_versus_ref"] = sample_var_ratio_ref_vs_q_pw.loc[nhood_diffs.index]

In [ ]:
nhood_diffs.head(5)

In [ ]:
from scipy.stats import pearsonr

In [ ]:
x = nhood_diffs["delta_mapqc_vs_alt"]
y = nhood_diffs["q_var_versus_ref"]
plt.scatter(x, y, s=1)
plt.xlabel("Delta mapqc score (pairwise - sinkhorn)")
plt.ylabel("(q sample variance) / (ref sample variance)")
plt.show()
print(pearsonr(x, y))

In [ ]:
mapqc.pl.scatter_nhood_latent_dims(res_pw, nhood_oi, "q_sample", dotsize=1)

### Results pairwise versus centroid:

In [ ]:
mapqc.pl.umap_mapqc_scores(res_pw)

In [ ]:
mapqc.pl.umap_mapqc_scores(res_alt)

binarized mapqc score:

In [ ]:
mapqc.pl.umap_mapqc_scores(res_pw, color_by="mapqc_score>2")

In [ ]:
mapqc.pl.umap_mapqc_scores(res_alt, color_by="mapqc_score>2")

In [ ]:
len(np.array(["a", "b", "d"]))

In [ ]:
import numpy as np

In [ ]:
res_pw.adata.obs

In [ ]:
center_cells = np.array(
    [
        "CTCGAAACACGCCAGT_GRO-09_biopsy",
        "CCTTCCCAGTGAACAT-1-4",
        "CATGCCTAGAAGATTC_F02611",
    ]
)

In [ ]:
center_cells

In [ ]:
center_cell_idc = np.where(res_pw.adata.obs.index.isin(center_cells))[0]
center_cell_idc = center_cell_idc[np.argsort(center_cells)]

In [ ]:
center_cell_idc